In [0]:
# ============================================================
# PFIN | Phase 3 | Gold Aggregation
# Notebook:  03_aggregate_gold
# Source:    pfin_dev.silver.{contributions, contributors,
#            recipients, electoral_events}
# Target:    pfin_dev.gold.{contributions_by_party,
#            contributions_by_district,
#            contributions_by_contributor_type,
#            contributions_by_year,
#            top_contributors}
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────────────
CATALOG        = "pfin_dev"
SILVER         = f"{CATALOG}.silver"
GOLD           = f"{CATALOG}.gold"
OPS            = f"{CATALOG}.ops"
TOP_N          = 100   # top contributors per fiscal year

TABLES = {
    "by_party":           f"{GOLD}.contributions_by_party",
    "by_district":        f"{GOLD}.contributions_by_district",
    "by_contributor_type":f"{GOLD}.contributions_by_contributor_type",
    "by_year":            f"{GOLD}.contributions_by_year",
    "top_contributors":   f"{GOLD}.top_contributors",
}

run_start = datetime.now()
print(f"[{run_start}] Phase 3 — Gold aggregation started")
print(f"  Catalog : {CATALOG}")
print(f"  Target  : {GOLD}")
print("-" * 60)


# ============================================================
# STEP 1: READ SILVER TABLES
# ============================================================
print(f"\n[{datetime.now()}] Reading Silver tables...")

df_contributions  = spark.table(f"{SILVER}.contributions")
df_contributors   = spark.table(f"{SILVER}.contributors")
df_recipients     = spark.table(f"{SILVER}.recipients")
df_events         = spark.table(f"{SILVER}.electoral_events")

# Quick counts
for name, df in [
    ("contributions",   df_contributions),
    ("contributors",    df_contributors),
    ("recipients",      df_recipients),
    ("electoral_events",df_events),
]:
    print(f"  silver.{name}: {df.count():,} rows")

print()


# ============================================================
# STEP 2: BUILD BASE FACT — contributions enriched
# ============================================================
# Join contributions with recipients and events.
# Contributors joined separately per Gold table to avoid
# pulling unnecessary columns into every aggregation.

df_fact = (
    df_contributions
    .join(
        df_recipients.select(
            "recipient_id", "political_party", "electoral_district", "political_entity"
        ),
        on="recipient_id", how="left"
    )
    .join(
        df_events.select("electoral_event_key", "fiscal_year"),
        on="electoral_event_key", how="left"
    )
)

# ── Fact with contributor type ────────────────────────────
df_fact_with_contributor = (
    df_fact
    .join(
        df_contributors.select("contributor_key", "contributor_type",
                               "contributor_name", "contributor_province"),
        on="contributor_key", how="left"
    )
)

NOW = F.lit(datetime.now().isoformat()).cast("timestamp")


# ============================================================
# STEP 3: contributions_by_party
# ============================================================
print(f"[{datetime.now()}] Building contributions_by_party...")

df_by_party = (
    df_fact
    .groupBy("political_party", "fiscal_year")
    .agg(
        F.sum("monetary_amount").cast("decimal(14,2)").alias("total_monetary"),
        F.sum("non_monetary_amount").cast("decimal(14,2)").alias("total_non_monetary"),
        F.sum("total_amount").cast("decimal(14,2)").alias("total_amount"),
        F.count("*").alias("contribution_count"),
        F.countDistinct("contributor_key").alias("unique_contributors"),
        F.avg("monetary_amount").cast("decimal(14,2)").alias("avg_contribution"),
    )
    .withColumn("_aggregated_at", NOW)
    .orderBy("fiscal_year", "total_amount", ascending=[True, False])
)

(df_by_party.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["by_party"]))

spark.sql(f"ALTER TABLE {TABLES['by_party']} CLUSTER BY (fiscal_year, political_party)")
spark.sql(f"ALTER TABLE {TABLES['by_party']} ENABLE PREDICTIVE OPTIMIZATION")
print(f"  contributions_by_party: {spark.table(TABLES['by_party']).count():,} rows ✓")


# ============================================================
# STEP 4: contributions_by_district
# ============================================================
print(f"[{datetime.now()}] Building contributions_by_district...")

df_by_district = (
    df_fact
    .groupBy("electoral_district", "fiscal_year", "political_party")
    .agg(
        F.sum("monetary_amount").cast("decimal(14,2)").alias("total_monetary"),
        F.sum("total_amount").cast("decimal(14,2)").alias("total_amount"),
        F.count("*").alias("contribution_count"),
        F.countDistinct("contributor_key").alias("unique_contributors"),
    )
    .withColumn("_aggregated_at", NOW)
    .orderBy("fiscal_year", "total_amount", ascending=[True, False])
)

(df_by_district.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["by_district"]))

spark.sql(f"ALTER TABLE {TABLES['by_district']} CLUSTER BY (fiscal_year, electoral_district)")
spark.sql(f"ALTER TABLE {TABLES['by_district']} ENABLE PREDICTIVE OPTIMIZATION")
print(f"  contributions_by_district: {spark.table(TABLES['by_district']).count():,} rows ✓")


# ============================================================
# STEP 5: contributions_by_contributor_type
# ============================================================
print(f"[{datetime.now()}] Building contributions_by_contributor_type...")

df_year_totals = (
    df_fact_with_contributor
    .groupBy("fiscal_year")
    .agg(F.sum("total_amount").alias("year_total"))
)

df_by_type = (
    df_fact_with_contributor
    .groupBy("contributor_type", "fiscal_year")
    .agg(
        F.sum("monetary_amount").cast("decimal(14,2)").alias("total_monetary"),
        F.sum("total_amount").cast("decimal(14,2)").alias("total_amount"),
        F.count("*").alias("contribution_count"),
        F.countDistinct("contributor_key").alias("unique_contributors"),
    )
    .join(df_year_totals, on="fiscal_year", how="left")
    .withColumn(
        "pct_of_total_amount",
        (F.col("total_amount") / F.col("year_total") * 100).cast("decimal(5,2)")
    )
    .drop("year_total")
    .withColumn("_aggregated_at", NOW)
    .orderBy("fiscal_year", "total_amount", ascending=[True, False])
)

(df_by_type.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["by_contributor_type"]))

spark.sql(f"ALTER TABLE {TABLES['by_contributor_type']} CLUSTER BY (fiscal_year, contributor_type)")
spark.sql(f"ALTER TABLE {TABLES['by_contributor_type']} ENABLE PREDICTIVE OPTIMIZATION")
print(f"  contributions_by_contributor_type: {spark.table(TABLES['by_contributor_type']).count():,} rows ✓")


# ============================================================
# STEP 6: contributions_by_year
# ============================================================
print(f"[{datetime.now()}] Building contributions_by_year...")

df_by_year = (
    df_fact
    .groupBy("fiscal_year")
    .agg(
        F.sum("monetary_amount").cast("decimal(14,2)").alias("total_monetary"),
        F.sum("non_monetary_amount").cast("decimal(14,2)").alias("total_non_monetary"),
        F.sum("total_amount").cast("decimal(14,2)").alias("total_amount"),
        F.count("*").alias("contribution_count"),
        F.countDistinct("contributor_key").alias("unique_contributors"),
        F.countDistinct("recipient_id").alias("unique_recipients"),
        F.avg("monetary_amount").cast("decimal(14,2)").alias("avg_contribution"),
        F.percentile_approx("monetary_amount", 0.5).cast("decimal(14,2)").alias("median_contribution"),
    )
    .withColumn("_aggregated_at", NOW)
    .orderBy("fiscal_year")
)

(df_by_year.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["by_year"]))

spark.sql(f"ALTER TABLE {TABLES['by_year']} CLUSTER BY (fiscal_year)")
spark.sql(f"ALTER TABLE {TABLES['by_year']} ENABLE PREDICTIVE OPTIMIZATION")
print(f"  contributions_by_year: {spark.table(TABLES['by_year']).count():,} rows ✓")


# ============================================================
# STEP 7: top_contributors
# ============================================================
print(f"[{datetime.now()}] Building top_contributors (top {TOP_N} per year)...")

from pyspark.sql.window import Window

df_top_base = (
    df_fact_with_contributor
    .groupBy("contributor_key", "contributor_name", "contributor_type",
             "contributor_province", "fiscal_year")
    .agg(
        F.sum("monetary_amount").cast("decimal(14,2)").alias("total_monetary"),
        F.sum("total_amount").cast("decimal(14,2)").alias("total_amount"),
        F.count("*").alias("contribution_count"),
        F.concat_ws(", ",
            F.collect_set(
                F.when(F.col("political_party").isNotNull(), F.col("political_party"))
            )
        ).alias("parties_contributed_to"),
    )
)

window_rank = Window.partitionBy("fiscal_year").orderBy(F.desc("total_amount"))

df_top = (
    df_top_base
    .withColumn("rank_in_year", F.row_number().over(window_rank))
    .filter(F.col("rank_in_year") <= TOP_N)
    .withColumn("_aggregated_at", NOW)
    .orderBy("fiscal_year", "rank_in_year")
)

(df_top.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["top_contributors"]))

spark.sql(f"ALTER TABLE {TABLES['top_contributors']} CLUSTER BY (fiscal_year, rank_in_year)")
spark.sql(f"ALTER TABLE {TABLES['top_contributors']} ENABLE PREDICTIVE OPTIMIZATION")
print(f"  top_contributors: {spark.table(TABLES['top_contributors']).count():,} rows ✓")


# ============================================================
# STEP 8: DATA QUALITY CHECKS
# ============================================================
print(f"\n[{datetime.now()}] Running data quality checks...")

dq_results = []
all_passed = True

def dq_check(name, passed, detail=""):
    global all_passed
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  [{status}] {name}{' — ' + detail if detail else ''}")
    dq_results.append((name, status, detail, datetime.now().isoformat()))

# Check 1: All tables have rows
for label, tbl in TABLES.items():
    rc = spark.table(tbl).count()
    dq_check(f"Row count > 0: {label}", rc > 0, f"{rc:,} rows")

# Check 2: No NULL group-by keys in key tables
null_party = spark.table(TABLES["by_party"]).filter(F.col("political_party").isNull()).count()
dq_check("No NULL political_party in by_party", null_party == 0, f"{null_party} nulls")

null_yr = spark.table(TABLES["by_year"]).filter(F.col("fiscal_year").isNull()).count()
dq_check("No NULL fiscal_year in by_year", null_yr == 0, f"{null_yr} nulls")

# Check 3: Year totals consistent between by_party and by_year
total_by_party = (spark.table(TABLES["by_party"])
    .groupBy("fiscal_year").agg(F.sum("total_amount").alias("sum_party")))
total_by_year  = (spark.table(TABLES["by_year"])
    .select("fiscal_year", F.col("total_amount").alias("sum_year")))

mismatch = (
    total_by_party.join(total_by_year, on="fiscal_year", how="inner")
    .filter(F.abs(F.col("sum_party") - F.col("sum_year")) > 0.01)
    .count()
)
dq_check("Amount consistency: by_party totals match by_year", mismatch == 0,
         f"{mismatch} year(s) with mismatch > $0.01")

# Check 4: top_contributors rank uniqueness per year
dup_ranks = (spark.table(TABLES["top_contributors"])
    .groupBy("fiscal_year", "rank_in_year")
    .count()
    .filter(F.col("count") > 1)
    .count())
dq_check("No duplicate rank_in_year per fiscal_year", dup_ranks == 0,
         f"{dup_ranks} duplicates")

print(f"\n  {'✅ All checks passed' if all_passed else '❌ Some checks failed — review above'}")

# Write DQ results to ops
dq_schema = "check_name STRING, status STRING, detail STRING, checked_at STRING"
df_dq = spark.createDataFrame(dq_results, schema=dq_schema)
df_dq = df_dq.withColumn("phase", F.lit("phase_3")).withColumn("run_start", F.lit(run_start.isoformat()))

(df_dq.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{OPS}.data_quality_log"))

if not all_passed:
    raise Exception("Phase 3 data quality checks failed. See pfin_dev.ops.data_quality_log for details.")


# ============================================================
# STEP 9: SUMMARY
# ============================================================
run_end = datetime.now()
duration = (run_end - run_start).total_seconds()

print(f"\n{'='*60}")
print(f"Phase 3 complete in {duration:.1f}s")
print(f"{'='*60}")
print(f"{'Table':<42} {'Rows':>10}")
print(f"{'-'*52}")
for label, tbl in TABLES.items():
    rc = spark.table(tbl).count()
    print(f"  {tbl:<40} {rc:>10,}")
print(f"{'='*60}")


[2026-06-01 20:37:18.883457] Phase 3 — Gold aggregation started
  Catalog : pfin_dev
  Target  : pfin_dev.gold
------------------------------------------------------------

[2026-06-01 20:37:18.883817] Reading Silver tables...
  silver.contributions: 282,098 rows
  silver.contributors: 123,451 rows
  silver.recipients: 1,004 rows
  silver.electoral_events: 44 rows

[2026-06-01 20:37:20.008951] Building contributions_by_party...
  contributions_by_party: 17 rows ✓
[2026-06-01 20:37:23.731583] Building contributions_by_district...
  contributions_by_district: 893 rows ✓
[2026-06-01 20:37:26.882336] Building contributions_by_contributor_type...
  contributions_by_contributor_type: 2 rows ✓
[2026-06-01 20:37:30.824431] Building contributions_by_year...
  contributions_by_year: 2 rows ✓
[2026-06-01 20:37:34.156867] Building top_contributors (top 100 per year)...
  top_contributors: 200 rows ✓

[2026-06-01 20:37:38.059192] Running data quality checks...
  [PASS] Row count > 0: by_party — 17 